# Create Work Volumes (oxjob #1554)

Which book each Crossref chapter is in. A chapter is a work and its book is a work; `CreateWorksBase` puts the book on
the chapter as `volume {id, display_name}`.

```
openalex.crossref.crossref_deduplicated (raw: ISBNs, types)  --> crossref_volume_records
crossref_volume_records + location_work_ids                  --> work_volumes (one row per chapter location)
work_volumes                                                 --> CreateWorksBase.volume
```

Runs after `Map_Work_Ids` (both ends need work ids) and before `Works_Base`. Sizing and rule precision:
oxjobs `working/link-book-chapters-to-books/` (EXPLORE.md, DESIGN.md).

### Crossref book records and their parts, ISBNs normalized

In [ ]:
-- Crossref book records and their parts, with ISBNs normalized to ISBN-13 (oxjob #1554).
-- Walden's Crossref parse drops ISBNs, so they are read here from the raw feed. One row per Crossref record of a
-- book-ish type: containers (the books) and parts (chapters, sections, entries). Read by the next cell only.
CREATE OR REPLACE TABLE openalex.works.crossref_volume_records
CLUSTER BY (doi)
AS
WITH base AS (
  SELECT
    LOWER(native_id) AS doi,
    type AS cr_type,
    GET(title, 0) AS title,
    ARRAY_DISTINCT(FILTER(TRANSFORM(ISBN, x -> REGEXP_REPLACE(UPPER(x), '[^0-9X]', '')), s -> LENGTH(s) IN (10, 13))) AS raw_isbns
  FROM openalex.crossref.crossref_deduplicated
  WHERE type IN (
    -- containers
    'book', 'monograph', 'edited-book', 'reference-book', 'book-set',
    -- parts
    'book-chapter', 'book-section', 'book-part', 'book-track', 'reference-entry', 'other'
  )
)
SELECT
  doi,
  cr_type,
  cr_type IN ('book', 'monograph', 'edited-book', 'reference-book', 'book-set') AS is_container,
  title,
  -- ISBN-10 -> ISBN-13: prefix 978, drop the old check digit, recompute the EAN-13 one
  ARRAY_DISTINCT(FILTER(TRANSFORM(raw_isbns, s -> CASE
    WHEN LENGTH(s) = 13 THEN s
    WHEN LENGTH(s) = 10 THEN CONCAT('978', SUBSTR(s, 1, 9), CAST((10 - AGGREGATE(SEQUENCE(1, 12), 0,
      (acc, i) -> acc + CAST(SUBSTR(CONCAT('978', SUBSTR(s, 1, 9)), i, 1) AS INT) * (CASE WHEN i % 2 = 1 THEN 1 ELSE 3 END)) % 10) % 10 AS STRING))
  END), s -> s RLIKE '^97[89][0-9]{10}$')) AS isbns
FROM base

### Link each part to its book's work

In [ ]:
-- work_volumes (oxjob #1554): which book each Crossref chapter is in, as OpenAlex work ids.
-- A part (chapter, section, entry) links to a Crossref book record by, in order of trust:
--   doi_stem     the part's DOI minus its last -_. segment, or cut at the first -_. after the slash, is a book record
--                (10.4324/9780429440229-7 -> 10.4324/9780429440229; 10.1007/978-3-642-40760-4_49 -> ...-4)
--   isbn         a part ISBN names exactly one book record, else exactly one with the part's DOI prefix
--   doi_pattern  Oxford Academic-hosted presses (...003.NNNN -> ...001.0001) and Peter Lang (10.3726/<isbn>/N)
-- When the stem and the ISBN name different books, no link (a set's or series' ISBN on the part, or a junk stem).
-- Both ends are mapped through location_work_ids, so the book is the work its own Crossref record lands on (that
-- settles book DOIs that two works share). One row per part location; CreateWorksBase picks the primary location's.
CREATE OR REPLACE TABLE openalex.works.work_volumes
CLUSTER BY (work_id)
AS
WITH containers AS (
  SELECT doi, title, isbns FROM openalex.works.crossref_volume_records WHERE is_container
),
parts AS (
  SELECT doi, isbns,
    REGEXP_EXTRACT(doi, '^(.*)[-_.][^-_./]+$', 1) AS stem1,
    REGEXP_EXTRACT(doi, '^(.*?)[-_.][^/]*$', 1) AS stem2,
    CASE
      WHEN doi RLIKE '\\.003\\.[0-9]{4}$' THEN REGEXP_REPLACE(doi, '\\.003\\.[0-9]{4}$', '.001.0001')
      WHEN doi RLIKE '^10\\.3726/[^/]+/[0-9]+$' THEN REGEXP_REPLACE(doi, '/[0-9]+$', '')
    END AS pattern_key
  FROM openalex.works.crossref_volume_records
  WHERE NOT is_container
),
isbn_index AS (
  SELECT isbn, COLLECT_SET(doi) AS book_dois
  FROM (SELECT doi, EXPLODE(isbns) AS isbn FROM containers)
  GROUP BY isbn
),
isbn_candidates AS (
  SELECT p.doi, ARRAY_DISTINCT(FLATTEN(COLLECT_LIST(i.book_dois))) AS book_dois
  FROM (SELECT doi, EXPLODE(isbns) AS isbn FROM parts) p
  JOIN isbn_index i ON i.isbn = p.isbn
  GROUP BY p.doi
),
picks AS (
  SELECT
    p.doi,
    COALESCE(s1.doi, s2.doi) AS stem_book,
    CASE
      WHEN SIZE(c.book_dois) = 1 THEN c.book_dois[0]
      WHEN SIZE(FILTER(c.book_dois, b -> SUBSTRING_INDEX(b, '/', 1) = SUBSTRING_INDEX(p.doi, '/', 1))) = 1
        THEN FILTER(c.book_dois, b -> SUBSTRING_INDEX(b, '/', 1) = SUBSTRING_INDEX(p.doi, '/', 1))[0]
    END AS isbn_book,
    pt.doi AS pattern_book
  FROM parts p
  LEFT JOIN containers s1 ON s1.doi = p.stem1
  LEFT JOIN containers s2 ON s2.doi = p.stem2 AND p.stem2 <> p.stem1
  LEFT JOIN isbn_candidates c ON c.doi = p.doi
  LEFT JOIN containers pt ON pt.doi = p.pattern_key
),
links AS (
  SELECT doi,
    COALESCE(stem_book, isbn_book, pattern_book) AS book_doi,
    CASE
      WHEN stem_book IS NOT NULL AND isbn_book IS NOT NULL THEN 'isbn_and_doi_stem'
      WHEN stem_book IS NOT NULL THEN 'doi_stem'
      WHEN isbn_book IS NOT NULL THEN 'isbn'
      ELSE 'doi_pattern'
    END AS method
  FROM picks
  WHERE COALESCE(stem_book, isbn_book, pattern_book) IS NOT NULL
    AND NOT (stem_book IS NOT NULL AND isbn_book IS NOT NULL AND stem_book <> isbn_book)
)
SELECT
  part.work_id,
  l.doi AS native_id,
  book.work_id AS volume_work_id,
  l.book_doi AS volume_doi,
  c.title AS volume_display_name,
  l.method
FROM links l
JOIN openalex.works.location_work_ids part
  ON part.provenance = 'crossref' AND part.native_id_namespace = 'doi' AND part.native_id = l.doi
JOIN openalex.works.location_work_ids book
  ON book.provenance = 'crossref' AND book.native_id_namespace = 'doi' AND book.native_id = l.book_doi
JOIN containers c ON c.doi = l.book_doi
WHERE part.work_id IS NOT NULL AND book.work_id IS NOT NULL AND part.work_id <> book.work_id